### Correcting the groups to fix the 22/80 random groups not have similarity and forcing at least 1 valid similarity

In [1]:

import pandas as pd
import numpy as np
import random
import ast
from itertools import combinations

# Load the original groups
all_groups = pd.read_csv("../results/groups.csv")
#apply ast.literal_eval to convert the string representation of lists into actual lists
all_groups["users"] = all_groups["users"].apply(ast.literal_eval)
# Load the MovieLens training data
columns = ["user_id", "movie_id", "rating", "timestamp"]

train = pd.read_csv(
    "../data/ml-100k/u1.base",
    sep="\t",
    names=columns
)

print("loaded groups:", len(all_groups))
print("training ratings:", len(train))


loaded groups: 240
training ratings: 80000


In [2]:

# Create a matrix with users as rows and movies as columns
user_movie_matrix = train.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)

# Calculate Pearson similarity between users
user_similarity = user_movie_matrix.T.corr(
    method="pearson",
    min_periods=10
)

print("User similarity matrix:", user_similarity.shape)


User similarity matrix: (943, 943)


In [3]:

# Count valid similarity pairs in a group
def cnt_valid_pairs(users):
    return sum(
        pd.notna(user_similarity.loc[a, b])
        for a, b in combinations(users, 2)
    )

# Select only random groups
random_grps = all_groups[all_groups["group_type"] == "random"].copy()

# Count valid pairs for each random group
random_grps["valid_pairs"] = random_grps["users"].apply(cnt_valid_pairs)

# Separate valid and invalid groups
valid_random_grps = random_grps[random_grps["valid_pairs"] >= 1].copy()
invalid_random_grps = random_grps[random_grps["valid_pairs"] == 0].copy()

print("number of groups to keep:", len(valid_random_grps))
print("number of groups to replace:", len(invalid_random_grps))


number of groups to keep: 58
number of groups to replace: 22


In [9]:

# Find users already assigned to valid random groups
used_users = {
    user
    for group in valid_random_grps["users"]
    for user in group
}

# Find users available for replacement groups
av_users = sorted(set(user_similarity.index) - used_users)
# Generate replacement groups
rng = random.Random(42)
new_groups = []

while len(new_groups) < len(invalid_random_grps):
    group = rng.sample(av_users, 4)

    # Accept groups with at least one valid similarity pair
    if cnt_valid_pairs(group) >= 1:
        new_groups.append(group)

        # Prevent users from appearing in another random group
        av_users = [
            user for user in av_users if user not in group
        ]

print("Replacement groups:", len(new_groups))
print("First replacement:", new_groups[0])


Replacement groups: 22
First replacement: [871, 142, 35, 363]


In [10]:

# Copy the original 240 groups
fixed_groups = all_groups.copy()

# Replace the 22 invalid random groups
for idx, users in zip(invalid_random_grps.index, new_groups):
    fixed_groups.at[idx, "users"] = users

    # Calculate average similarity for the new group
    similarities = [
        user_similarity.loc[a, b]
        for a, b in combinations(users, 2)
    ]

    fixed_groups.at[idx, "avg_similarity"] = np.nanmean(similarities)

    # Old recommendation overlap is no longer valid
    fixed_groups.at[idx, "overlap"] = np.nan

# Record valid pair counts for every group
fixed_groups["valid_pairs"] = fixed_groups["users"].apply(cnt_valid_pairs)

print("Total groups:", len(fixed_groups))
print("Random groups missing similarity:",
      fixed_groups.loc[
          fixed_groups["group_type"] == "random",
          "avg_similarity"
      ].isna().sum())


Total groups: 240
Random groups missing similarity: 0


In [6]:


# Check that we have 240 groups
assert len(fixed_groups) == 240

# Check that every group has exactly 4 different users
assert fixed_groups["users"].apply(
    lambda users: len(users) == 4 and len(set(users)) == 4
).all()

# Check that users do not repeat within each category
for group_type, groups in fixed_groups.groupby("group_type"):
    users = [user for group in groups["users"] for user in group]
    assert len(users) == len(set(users)) == 320

# Check that all random groups have a similarity score
random_grps = fixed_groups[fixed_groups["group_type"] == "random"]
assert random_grps["avg_similarity"].notna().all()

print(" group tests passed!")


 group tests passed!


In [7]:

# Save the validated groups
fixed_groups.to_csv("../results/groups_revised.csv", index=False)

print("Fixed groups saved successfully! for now...")


Fixed groups saved successfully! for now...


In [8]:

from surprise import Dataset, Reader, SVD

# Prepare the original training data
reader = Reader(rating_scale=(1, 5))

data = Dataset.load_from_df(
    train[["user_id", "movie_id", "rating"]],
    reader
)

# Recreate our original SVD model
model = SVD(random_state=42)
model.fit(data.build_full_trainset())

# Check that the original prediction is reproduced
print("SVD prediction:", model.predict(556, 313).est)


SVD prediction: 4.559519027993886
